In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
bronze_customers_path = f"{BRONZE_PATH}/customers"
silver_customers_path = f"{SILVER_PATH}/customers"
pipeline_name = 'transform_customers_silver'
run_ts = datetime.now(timezone.utc)

In [0]:
watermark_record_df = spark.read \
    .format("delta") \
    .load(f"{METADATA_PATH}/silver_pipeline_watermarking") \
    .filter(F.col("pipeline_name") == pipeline_name) \
    .first()

In [0]:
watermark = (datetime(2000, 1, 1)) if watermark_record_df is None else watermark_record_df["last_processed_timestamp"]

In [0]:
df_customers_bronze = spark.read.format("delta") \
    .load(bronze_customers_path)

In [0]:
display(
    df_customers_bronze.orderBy(F.col("customer_id").asc(), F.col("updated_at").asc())
)

In [0]:
df_customers_silver = spark.read.format("delta") \
    .load(silver_customers_path)

In [0]:
is_silver_table_empty = df_customers_silver.isEmpty()

In [0]:
# To get rid of boundary records duplicated by the watermark due to data type mismatch between Sql Server and
# Databricks
if is_silver_table_empty:
    windowSpec = Window.partitionBy("customer_id", "updated_at").orderBy(F.col("updated_at").desc())

    df_deduped_bronze_customers = df_customers_bronze \
        .withColumn(
            "rn",
            F.row_number().over(windowSpec)
        ) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # display(df_deduped_bronze_customers)

In [0]:
# There was already data in bronze. There were customers for which there were more than one updated record in Bronze already.
if is_silver_table_empty:
    windowSpec = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
    df_deduped_bronze_customers = df_deduped_bronze_customers \
        .withColumn(
            "valid_from",
            F.col("updated_at")
        ) \
        .withColumn(
            "valid_to",
            F.lag(F.col("updated_at"), offset = 1, default = None).over(windowSpec)
        ) \
        .withColumn(
            "rn",
            F.row_number().over(windowSpec)
        ) \
        .withColumn(
            "is_current",
            F.when(F.col("rn") == 1, True).otherwise(False)
        ) \
        .drop("rn")

    # display(df_deduped_bronze_customers.filter(~F.col("is_current")))

In [0]:
if is_silver_table_empty:
    df_deduped_bronze_customers.write.format("delta").mode("append").save(silver_customers_path)
    # display(spark.read.format("delta").load(silver_customers_path))

In [0]:
# The code in this cell assumes that there cannot be more than one updated record for the same customer when
# this pipeline runs. It keeps only the latest record for each customer
if not is_silver_table_empty:
    windowSpec = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())

    df_deduped_bronze_customers = df_customers_bronze \
        .withColumn(
            "rn",
            F.row_number().over(windowSpec)
        ) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # display(df_deduped_bronze_customers)

In [0]:
if not is_silver_table_empty:
    df_current_silver_records = df_customers_silver \
        .filter(F.col("is_current"))

In [0]:
if not is_silver_table_empty:
    display(df_current_silver_records)

In [0]:
if not is_silver_table_empty:
    df_changed_customers = df_deduped_bronze_customers.alias("bronze").join(
        df_current_silver_records.alias("current_silver"),
        F.col("bronze.customer_id") == F.col("current_silver.customer_id"),
    ) \
        .filter(F.col("bronze.phone") != F.col("current_silver.phone")) \
        .select(
            "bronze.*"
        )

In [0]:
if not is_silver_table_empty:
    display(df_changed_customers)

In [0]:
if not is_silver_table_empty:
    df_new_customers = df_deduped_bronze_customers.alias("bronze").join(
        df_current_silver_records.alias("current"),
        F.col("bronze.customer_id") == F.col("current.customer_id"),
        "left_anti"
    )

In [0]:
if not is_silver_table_empty:
    display(df_new_customers)

In [0]:
# Close current silver records for customers whose pone changed.
if not is_silver_table_empty:
    from pyspark.sql.types import *
    from delta.tables import DeltaTable

    silver_table = DeltaTable.forPath(
        spark,
        silver_customers_path
    )

    silver_table.alias("target").merge(
        df_changed_customers.alias("source"),
        "source.customer_id = target.customer_id and target.is_current"
    ) \
        .whenMatchedUpdate(
            set = {
                "valid_to": "source.updated_at",
                "is_current": F.lit(False).cast(BooleanType())
            }
        ) \
        .execute()


In [0]:
if not is_silver_table_empty:
    display(
        spark.read.format("delta").load(silver_customers_path)
    )

In [0]:
# Customer records to insert into silver
if not is_silver_table_empty:
    df_records_to_insert = df_changed_customers.union(df_new_customers)

In [0]:
if not is_silver_table_empty:
    display(df_records_to_insert)

In [0]:
if not is_silver_table_empty:
    from pyspark.sql.types import *

    df_records_to_insert = df_records_to_insert \
        .withColumn(
            "valid_from",
            F.col("updated_at")
        ) \
        .withColumn(
            "valid_to",
            F.lit(None).cast(TimestampType())
        ) \
        .withColumn(
            "is_current",
            F.lit(True).cast(BooleanType())
        )

In [0]:
if not is_silver_table_empty:
    df_records_to_insert.write.format("delta").mode("append").save(silver_customers_path)

In [0]:
if not is_silver_table_empty:
    display(
        spark.read.format("delta") \
            .load(f"{SILVER_PATH}/customers") \
            .filter(F.col("is_current")) \
            .orderBy(F.col("customer_id").asc()))